# Nove dos dez relatórios com `Onychomycosis` são um paciente só

O ranking apontou para micose de unha. Este notebook segue os pares de
volta até os documentos que os produziram.

In [1]:
import sys

sys.path.insert(0, "../src")

import duckdb
import pandas

from hindsight.analysis.crowding import breadth, overlap, wide_reports

PARTITION = "../data/parquet/year=2025/quarter=1/part=0001-of-0028"

In [2]:
carriers = duckdb.sql(f'''
    SELECT r.safetyreportid, count(DISTINCT d.medicinalproduct) AS drugs
    FROM '{PARTITION}/report_reaction.parquet' AS r
    JOIN '{PARTITION}/report_drug.parquet' AS d USING (safetyreportid)
    WHERE r.reactionmeddrapt = 'Onychomycosis'
    GROUP BY 1
    ORDER BY 2 DESC
''').df()

carriers

,safetyreportid,drugs
0,25021632,96
1,25044641,91
2,23733354,89
3,24757080,84
4,24965286,78
5,24361485,77
6,24087637,73
7,24816316,73
8,24388883,66
9,20836166,3


Dez relatórios carregam o termo. Nove deles nomeiam entre 66 e 96
medicamentos distintos; o décimo nomeia três e é um relatório comum.

Um relatório que nomeia 90 medicamentos e 10 eventos afirma 900 pares
medicamento–evento. Nove desses colocam `a = 9` em cada um deles.

In [3]:
cluster = carriers[carriers.drugs > 10].safetyreportid.tolist()

scores = pandas.DataFrame(
    overlap(cluster, root="../data/parquet"),
    columns=["report", "other", "jaccard"],
)

scores.jaccard.describe()[["min", "50%", "max"]]

min    0.376238
50%    0.480762
max    0.908163
Name: jaccard, dtype: float64

## Jaccard 0,38 mínimo, 0,48 mediana, 0,91 máximo

Não são nove pacientes que por acaso tomam muitos medicamentos. Dois
deles compartilham um `companynumb` e uma lista idêntica de dez
reações — o mesmo caso, registrado duas vezes.

In [4]:
duckdb.sql(f'''
    SELECT safetyreportid, companynumb, occurcountry, serious, receiptdate
    FROM '{PARTITION}/report.parquet'
    WHERE safetyreportid IN {tuple(cluster)}
    ORDER BY companynumb
''')

┌────────────────┬───────────────────────────────────────────┬──────────────┬─────────┬─────────────┐
│ safetyreportid │                companynumb                │ occurcountry │ serious │ receiptdate │
│    varchar     │                  varchar                  │   varchar    │ varchar │   varchar   │
├────────────────┼───────────────────────────────────────────┼──────────────┼─────────┼─────────────┤
│ 24757080       │ CA-BEH-2024187690                         │ CA           │ 1       │ 20250311    │
│ 24965286       │ CA-BIOCON BIOLOGICS LIMITED-BBL2025000590 │ CA           │ 1       │ 20250213    │
│ 24087637       │ CA-JNJFOC-20240708638                     │ CA           │ 1       │ 20250319    │
│ 24361485       │ CA-JNJFOC-20240931166                     │ CA           │ 1       │ 20250217    │
│ 23733354       │ CA-PURDUE-USA-2024-0308932                │ CA           │ 1       │ 20250318    │
│ 24388883       │ CA-PURDUE-USA-2024-0312263                │ CA           │ 1   

Todos canadenses, todos graves, registrados entre janeiro e março de
2025 por seis fabricantes diferentes — cada empresa cujo produto
estava na lista reportou de forma independente.

Duas correções parecem óbvias e nenhuma funciona.

In [5]:
duckdb.sql(f'''
    SELECT drugcharacterization, count(*) AS rows
    FROM '{PARTITION}/report_drug.parquet'
    WHERE safetyreportid IN {tuple(cluster)}
    GROUP BY 1
''')

┌──────────────────────┬───────┐
│ drugcharacterization │ rows  │
│       varchar        │ int64 │
├──────────────────────┼───────┤
│ 2                    │    29 │
│ 1                    │  2345 │
└──────────────────────┴───────┘

Por convenção a desproporcionalidade roda só sobre medicamentos
suspeitos. Aqui **todos** os medicamentos desses relatórios estão
marcados como suspeitos, então `drugcharacterization` não remove
nada. O critério de triagem também não — o notebook 02 mediu isso.

O que separa é o formato do documento.

In [6]:
breadth(root="../data/parquet")

{'cut': 27.0, 'median': 2.0, 'widest': 121, 'reports': 12000}

## O relatório mediano nomeia dois medicamentos; o percentil 99 nomeia 27

O cluster nomeia de 66 a 96. Essa distância é o que faz de um quantil
um corte utilizável em vez de esperançoso — nada fica em cima da linha
discutindo de que lado pertence.

Uma constante não viajaria: isto é uma partição de um export, e o
corpus vai de 2004 a 2025.

In [7]:
wide = wide_reports(cut=27, root="../data/parquet")

len(wide), len(wide) / 12000

(125, 0.010416666666666666)

125 relatórios de 12.000 — 1,04%. O que esse 1% faz com a tabela de
pares é o resultado do M0, e está em `reports/m0.qmd`.

**Isto não é uma regra de deduplicação.** Decidir que dois relatórios
descrevem um caso só exige resolução de entidades, que é o M2. Uma
lotação alta diz que a evidência não distingue um caso repetido de um
real — não que o par seja falso. Um paciente com 90 medicamentos que
tem uma reação adversa é um paciente real.